# The Waveform Generator Zoo

A visual + audible field guide to CLM's oscillator family: polynomial generators, sine-summation variants, band-limited waveforms, and wavetables -- each plotted as waveform and spectrum.

Source: [CLM generators](https://ccrma.stanford.edu/software/snd/snd/sndclm.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm

SR = 44100

def spectrum(x, n=4096):
    rl = np.copy(x[:n].astype(float)); im = np.zeros(n)
    clm.mus_fft(rl, im, n, 1)
    clm.mus_rectangular2polar(rl, im)
    return np.abs(rl[:n//2])

def wave_and_spectrum(name, gen, dur=0.05, n=4096):
    samples = np.array([gen() for _ in range(n)])
    plt.figure(figsize=(14, 4))
    plt.subplot(1, 2, 1)
    plt.plot(samples[:400], lw=0.8); plt.title(name)
    plt.subplot(1, 2, 2)
    plt.plot(np.arange(n//2) * SR / n, spectrum(samples), lw=0.8)
    plt.xlim(0, 10000); plt.title(f'{name}: spectrum')
    plt.tight_layout(); plt.show()
    return samples

## Band-limited classics

`sawtooth_wave`, `square_wave` and `triangle_wave` are band-limited (they sum sine partials rather than switching abruptly), so they do not alias at high frequencies:

In [ ]:
wave_and_spectrum('triangle', clm.make_triangle_wave(220.0))
wave_and_spectrum('sawtooth', clm.make_sawtooth_wave(220.0))
wave_and_spectrum('square', clm.make_square_wave(220.0))

## Polynomial generators

`polywave` and `polyshape` evaluate polynomial partials (Chebyshev-style) -- the "partials" list pairs harmonic index and coefficient. This is waveshaping by another route:

In [ ]:
wave_and_spectrum('polywave [0,1,3,2,5,1]',
                  clm.make_polywave(220.0, [0.0, 1.0, 3.0, 2.0, 5.0, 1.0]))

In [ ]:
wave_and_spectrum('polyshape', clm.make_polyshape(220.0))

## The n-sine and n-cosine families

These sum n harmonically-related sines/cosines with various weightings (`ncos`, `nsin`, `nrxysin`, `nrxycos`...). The `rxy` variants take a ratio and a radius:

In [ ]:
wave_and_spectrum('ncos n=3', clm.make_ncos(220.0, 3))
wave_and_spectrum('nsin n=4', clm.make_nsin(220.0, 4))
wave_and_spectrum('nrxysin', clm.make_nrxysin(220.0))
wave_and_spectrum('nrxycos', clm.make_nrxycos(220.0))

## Wavetables

`table_lookup` plays any single-cycle table. `partials2wave` builds tables from harmonic/amplitude pairs -- with the same pairs producing "clarinet" (odd harmonics) vs "reed" (even-ish) flavors:

In [ ]:
clar = clm.partials2wave([1, 1, 3, 0.5, 5, 0.25, 7, 0.12])   # odd harmonics
reed = clm.partials2wave([1, 1, 2, 0.7, 3, 0.3, 4, 0.1])     # even + odd
wave_and_spectrum('table: odd partials', clm.make_table_lookup(220.0, wave=clar))
wave_and_spectrum('table: all partials', clm.make_table_lookup(220.0, wave=reed))

In [ ]:
with clm.Sound('clarinet-ish.wav', channels=1, srate=SR):
    t = clm.make_table_lookup(220.0, wave=clar)
    amp = clm.make_env([0, 0, 10, 1, 90, 1, 100, 0], duration=1.0)
    for i in range(SR):
        clm.outa(i, clm.env(amp) * clm.table_lookup(t))

In [ ]:
ipd.Audio('clarinet-ish.wav', rate=SR)

## Summary

- All the classic shapes exist as band-limited generators (no aliasing)
- Polynomial generators (`polywave`/`polyshape`) are compact spectral recipes
- Wavetables via `partials2wave` remain the most flexible option

Next: [noise and random signals](10_Noise.ipynb)